In [5]:
import numpy as np
import pandas as pd
import joblib
import requests
from keras.models import load_model

SEQ_LEN = 24
LATITUDE = 39
LONGITUDE = 35
DEFAULT_CITY = "Izmir"

MODEL_PATH = "../energy_model_final.keras"
SCALER_X_PATH = "../scaler_x.pkl"
SCALER_Y_PATH = "../scaler_y.pkl"

model = load_model(MODEL_PATH)
scaler_x = joblib.load(SCALER_X_PATH)
scaler_y = joblib.load(SCALER_Y_PATH)
FEATURES = list(scaler_x.feature_names_in_)

WEEKDAY_MAP = {"Monday":0,"Tuesday":1,"Wednesday":2,"Thursday":3,"Friday":4,"Saturday":5,"Sunday":6}
LAGS = [1,2,3,6,12,24,48,72,168,336]
WINDOWS = [6,12,24,48,168]
ROLL_MINMAX_WINDOWS = [6,24,168]

def load_history(csv_path, city=None):
    df = pd.read_csv(csv_path)
    df["Timestamp"] = pd.to_datetime(df["Timestamp"])
    if "city" not in df.columns:
        df["city"] = city if city else DEFAULT_CITY
    elif city:
        df = df[df["city"]==city].copy()
    df = df.sort_values("Timestamp").drop_duplicates("Timestamp").reset_index(drop=True)
    return df

def fetch_weather(start_ts, end_ts):
    start_date = pd.to_datetime(start_ts).strftime("%Y-%m-%d")
    end_date = pd.to_datetime(end_ts).strftime("%Y-%m-%d")
    params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "start_date": start_date,
        "end_date": end_date,
        "hourly":["temperature_2m","dew_point_2m","precipitation","windspeed_10m"],
        "timezone":"auto"
    }
    url = "https://archive-api.open-meteo.com/v1/archive"
    r = requests.get(url, params=params, timeout=60)
    if r.status_code != 200:
        url = "https://api.open-meteo.com/v1/forecast"
        r = requests.get(url, params=params, timeout=60)
    data = r.json().get("hourly", {})
    if not data or "time" not in data:
        return pd.DataFrame()
    df = pd.DataFrame({
        "Timestamp": pd.to_datetime(data["time"]),
        "tavg": np.array(data.get("temperature_2m", []), dtype=float),
        "prcp": np.array(data.get("precipitation", []), dtype=float),
        "wspd": np.array(data.get("windspeed_10m", []), dtype=float),
        "d2m": np.array(data.get("dew_point_2m", []), dtype=float)
    })
    t, d = df["tavg"], df["d2m"]
    alpha_d = (17.625*d)/(243.04+d)
    alpha_t = (17.625*t)/(243.04+t)
    df["humidity"] = 100*np.exp(alpha_d-alpha_t).clip(0,100)
    return df[["Timestamp","tavg","prcp","wspd","humidity"]]

def build_features(df, city_name=None):
    df = df.copy()
    df["Timestamp"] = pd.to_datetime(df["Timestamp"])
    df = df.sort_values("Timestamp").reset_index(drop=True)
    if city_name is None:
        city_name = df["city"].iloc[0] if "city" in df.columns else DEFAULT_CITY
    if "city" not in df.columns:
        df["city"] = city_name
    df["EnergyConsumption"] = df.get("EnergyConsumption", np.nan).ffill()
    df["Energy_log"] = np.log1p(df["EnergyConsumption"])
    df["hour"] = df["Timestamp"].dt.hour
    df["weekday_num"] = df["Timestamp"].dt.weekday
    df["month"] = df["Timestamp"].dt.month
    df["dayofyear"] = df["Timestamp"].dt.dayofyear
    df["year"] = df["Timestamp"].dt.year
    df["is_holiday"] = df["weekday_num"].isin([5,6]).astype(int)
    df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
    df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)
    df["month_sin"] = np.sin(2*np.pi*df["month"]/12)
    df["month_cos"] = np.cos(2*np.pi*df["month"]/12)
    hour_week = df["hour"] + df["weekday_num"]*24
    df["hour_week_sin"] = np.sin(2*np.pi*hour_week/168)
    df["hour_week_cos"] = np.cos(2*np.pi*hour_week/168)
    for c in ["tavg","humidity","wspd","prcp"]:
        if c not in df.columns:
            df[c] = 0
        df[c] = df[c].ffill().bfill()
    df["temp_sq"] = df["tavg"]**2
    df["temp_humidity"] = df["tavg"]*df["humidity"]
    df["temp_wind"] = df["tavg"]*df["wspd"]
    df["wind_humidity"] = df["wspd"]*df["humidity"]
    for l in LAGS: df[f"lag_{l}"] = df["Energy_log"].shift(l)
    shifted = df["Energy_log"].shift(1)
    for w in WINDOWS:
        df[f"roll_mean_{w}"] = shifted.rolling(w).mean()
        df[f"roll_std_{w}"] = shifted.rolling(w).std()
    for w in ROLL_MINMAX_WINDOWS:
        df[f"roll_min_{w}"] = shifted.rolling(w).min()
        df[f"roll_max_{w}"] = shifted.rolling(w).max()
    df["diff_1"] = df["Energy_log"].diff(1)
    df["diff_24"] = df["Energy_log"].diff(24)
    city_cols = [c for c in FEATURES if c.startswith("city_")]
    for col in city_cols: df[col]=0
    if f"city_{city_name}" in city_cols: df[f"city_{city_name}"]=1
    for i in range(7): df[f"dow_{i}"] = (df["weekday_num"]==i).astype(int)
    for col in FEATURES:
        if col not in df.columns:
            df[col] = 0
    return df.reset_index(drop=True)

def predict_hour_by_hour(df, target_ts, city_name):
    df = df.copy()
    df["Timestamp"] = pd.to_datetime(df["Timestamp"])
    target_ts = pd.to_datetime(target_ts)
    if target_ts <= df["Timestamp"].max():
        feat_df = build_features(df, city_name)
        i = feat_df.index[feat_df["Timestamp"]==target_ts][0]
        if i < SEQ_LEN:
            raise ValueError(f"Not enough past data for prediction at {target_ts}")
        X = scaler_x.transform(feat_df.loc[i-SEQ_LEN:i-1, FEATURES])
        seq = X.reshape(1, SEQ_LEN, len(FEATURES))
        pred_scaled = model.predict(seq, verbose=0)
        pred_log = scaler_y.inverse_transform(pred_scaled)
        df.loc[df["Timestamp"]==target_ts,"EnergyConsumption"] = float(np.expm1(pred_log[0][0]))
        return df.loc[df["Timestamp"]==target_ts,"EnergyConsumption"].values[0], df
    last_hist_ts = df["Timestamp"].max()
    full_range = pd.date_range(start=last_hist_ts+pd.Timedelta(hours=1), end=target_ts, freq="h")
    weather_df = fetch_weather(full_range[0], full_range[-1])
    if weather_df.empty:
        last_weather = df[["tavg","humidity","wspd","prcp"]].iloc[-1]
        weather_df = pd.DataFrame({
            "Timestamp": full_range,
            "tavg": last_weather["tavg"],
            "humidity": last_weather["humidity"],
            "wspd": last_weather["wspd"],
            "prcp": last_weather["prcp"]
        })
    df = pd.concat([df, weather_df], ignore_index=True)
    df = df.drop_duplicates("Timestamp").sort_values("Timestamp").reset_index(drop=True)
    for ts in full_range:
        feat_df = build_features(df, city_name)
        i = feat_df.index[feat_df["Timestamp"]==ts][0]
        if i < SEQ_LEN:
            continue
        X = scaler_x.transform(feat_df.loc[i-SEQ_LEN:i-1, FEATURES])
        seq = X.reshape(1, SEQ_LEN, len(FEATURES))
        pred_scaled = model.predict(seq, verbose=0)
        pred_log = scaler_y.inverse_transform(pred_scaled)
        pred = float(np.expm1(pred_log[0][0]))
        df.loc[df["Timestamp"]==ts,"EnergyConsumption"] = pred
    return df.loc[df["Timestamp"]==target_ts,"EnergyConsumption"].values[0], df

def predict_for_timestamp(csv_path, target_timestamp, city=None):
    df = load_history(csv_path, city)
    city_name = city if city else df["city"].iloc[0]
    pred, _ = predict_hour_by_hour(df, target_timestamp, city_name)
    return float(pred)

def predict_range(csv_path, start_ts, end_ts, city=None):
    df = load_history(csv_path, city)
    city_name = city if city else df["city"].iloc[0]
    start_ts = pd.to_datetime(start_ts)
    end_ts = pd.to_datetime(end_ts)
    full_range = pd.date_range(start=start_ts, end=end_ts, freq="h")
    results = []
    df_working = df.copy()
    for ts in full_range:
        pred, df_working = predict_hour_by_hour(df_working, ts, city_name)
        results.append({"Timestamp": ts, "PredictedEnergy": float(pred)})
    return pd.DataFrame(results)

if __name__=="__main__":
    
    pred = predict_for_timestamp("consumptions.csv","2026-04-16 10:00","Izmir")#returns consumption
    print(pred)


    #df_pred = predict_range("consumptions.csv","2026-03-25 02:00","2026-03-25 12:00","Izmir")#returns a DF
    #print(df_pred)

46642.859375
